# Python 基礎（第 1 本）：讀資料・算數字・畫圖

> **目標**：學會三本 ML notebook 開頭都會用到的純 Python 功夫——
> `import`、用 `pandas` 讀表格、用 `numpy` 算數學、用 `matplotlib` 畫圖。
> **不碰** class 和神經網路（那是第 2 本 `ML_教學.ipynb`）。

用法：由上往下，每格 `Shift+Enter`。本書用真實資料示範，整本可執行。

學習路線：
- **第 1 本（本檔）**：Python 基礎 + 資料處理
- 第 2 本 `ML_教學.ipynb`：class / torch / 訓練迴圈
- 第 3 本 `PCA_AE_VAE_教學.ipynb`：三種壓縮法的概念與視覺化

## 0 · 環境自檢

In [ ]:
import importlib
for m in ['numpy','pandas','matplotlib']:
    mod = importlib.import_module(m)
    print('OK ', m, mod.__version__)

## 1 · `import`：把工具箱搬進來

`as np` 是取別名，少打字。`from X import Y` 只拿一個東西。

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 自動找 data 資料夾
CANDS = [
    '../../workshop_build/workshop_build/09_advanced_ml/gan/data',
    'workshop_build/workshop_build/09_advanced_ml/gan/data',
    'data', '../data', '../../gan/data',
]
DATA = None
for guess in CANDS:
    if os.path.exists(os.path.join(guess, 'stakeholder_relations.parquet')):
        DATA = guess; break
print('資料夾：', DATA)

df = pd.read_parquet(os.path.join(DATA, 'stakeholder_relations.parquet'))
print('資料形狀：', df.shape, '(列 = 街區, 欄 = 特徵)')
df.head()

In [ ]:
# 讓 matplotlib 顯示中文（找不到字型就略過，不影響程式）
from matplotlib import font_manager as fm
for fp in [
    '../../.venv/Lib/site-packages/matplotlib/mpl-data/fonts/ttf/NotoSansCJKsc-Regular.otf',
    '../../../.venv/Lib/site-packages/matplotlib/mpl-data/fonts/ttf/NotoSansCJKsc-Regular.otf',
    'C:/Windows/Fonts/msjh.ttc',
]:
    if os.path.exists(fp):
        try:
            fm.fontManager.addfont(fp)
            plt.rcParams['font.sans-serif'] = [fm.FontProperties(fname=fp).get_name()]
            break
        except Exception: pass
plt.rcParams['axes.unicode_minus'] = False

**拆解**
- `import numpy as np` / `pandas as pd` / `matplotlib.pyplot as plt`：載入並取別名。
- `os.path.join(a,b)`：安全接路徑。`for...if os.path.exists` 找到資料夾就 `break`。
- `pd.read_parquet(路徑)`：把檔讀成表格 `df`。`df.shape`=(列,欄)。`df.head()` 看前 5 列。

## 2 · 看懂表格 `DataFrame`

In [ ]:
print('共', len(df.columns), '欄：')
print(list(df.columns))
print('---')
print('每欄型別：'); print(df.dtypes)

## 3 · list 推導式：一行挑出要的欄位

```python
feat_cols = [c for c in df.columns if c not in drop and np.issubdtype(df[c].dtype, np.number)]
```
讀法：「對每個欄名 c，若不在 drop 裡且是數字欄，就留下」。

In [ ]:
# 先看最小例子
nums = [1,2,3,4,5]
print('平方：', [n*n for n in nums])
print('偶數：', [n for n in nums if n % 2 == 0])

In [ ]:
drop = ['site','gx','gy']
feat_cols = [c for c in df.columns
             if c not in drop and np.issubdtype(df[c].dtype, np.number)]
print('特徵欄 %d 個：' % len(feat_cols)); print(feat_cols)

X = df[feat_cols].to_numpy(dtype='float32')   # 表格 → 純數字陣列
print('X 形狀：', X.shape)

## 4 · `numpy` 陣列與標準化

numpy 讓你「對整個陣列一次運算」。標準化：每欄變平均 0、標準差 1。
```python
mu = X.mean(axis=0)   # 每欄平均
sd = X.std(axis=0)    # 每欄標準差
Xs = (X - mu) / sd    # 廣播：整表一次處理
```

In [ ]:
mu = X.mean(axis=0)
sd = X.std(axis=0); sd[sd < 1e-8] = 1.0   # 避免除以 0
Xs = (X - mu) / sd
print('標準化後：', Xs.shape)
print('每欄平均≈0：', Xs.mean(axis=0).round(2)[:5], '...')
print('每欄標準差≈1：', Xs.std(axis=0).round(2)[:5], '...')

## 5 · 布林遮罩與 `argmax`：圖上顏色哪來的

- `argmax(axis=1)`：每列找最大值的位置（0/1/2/3）。
- `dominant == i`：產生一排 True/False，用來只挑某一類。

In [ ]:
share_cols = ['share_state','share_developer','share_resident','share_unknown']
share_idx  = [feat_cols.index(c) for c in share_cols]
names  = ['state 政府','developer 開發商','resident 居民','unknown 未知']
colors = ['#1f77b4','#d62728','#2ca02c','#7f7f7f']

dominant = X[:, share_idx].argmax(axis=1)
print('前 10 個街區主導者：', dominant[:10])
print('state 主導的街區數：', (dominant == 0).sum())

## 6 · `def` 函式：把步驟打包

`def 名字(參數):` → 縮排內容 → `return` 交回結果。

In [ ]:
def standardize(arr):
    m = arr.mean(axis=0); s = arr.std(axis=0); s[s < 1e-8] = 1.0
    return (arr - m) / s

print('和上面一樣嗎：', np.allclose(Xs, standardize(X)))

## 7 · `matplotlib` 畫點圖

套路：**for 迴圈，一類一個顏色，疊在同一張圖**。這裡先用最簡單的 PCA 壓成 2 維來畫。

In [ ]:
from sklearn.decomposition import PCA
Z = PCA(n_components=2).fit_transform(Xs)   # (5438, 2)

plt.figure(figsize=(8,6))
for i in range(4):
    m = (dominant == i)
    plt.scatter(Z[m,0], Z[m,1], s=8, c=colors[i], label=names[i], alpha=0.6)
plt.xlabel('維度 1'); plt.ylabel('維度 2')
plt.title('5438 個街區壓成 2 維')
plt.legend(); plt.grid(alpha=0.3); plt.show()

**拆解（三本都一樣）**：`plt.figure()` 開圖 → `for i in range(4)` 四類 → `plt.scatter(x,y,c=,label=)` 畫點 → `plt.legend()`/`plt.show()`。改 `s=`、`alpha=` 看效果。

---
## 第 1 本完成
你會了：`import` / `pandas` 讀表 / list 推導式 / `numpy` 標準化 / 布林遮罩 / `def` / `matplotlib`。

**下一步**：打開 **`ML_教學.ipynb`**，學 `class` 和 `torch` 訓練迴圈。